# 03 Evaluation

This notebook covers model comparison, volatility regime analysis, feature importance, and backtesting.

In [ ]:
# Step 1: load generated artifacts from training and optional prediction exports.
from pathlib import Path
import json
import pandas as pd

from src.evaluation.pipeline import run_feature_importance_pipeline, run_volatility_and_backtest_pipeline

summary_path = Path('artifacts/results_summary.json')
pred_path = Path('artifacts/demo_predictions.csv')

summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
pred_df = pd.read_csv(pred_path) if pred_path.exists() else pd.DataFrame()
summary.keys(), pred_df.shape

In [ ]:
# Step 2: run feature importance pipeline (MI always; VSN/ablation optional).
if not pred_df.empty and 'label' in pred_df.columns:
    feature_cols = [c for c in pred_df.columns if c.startswith(('log_', 'hl_', 'oc_', 'upper_', 'volume_', 'rsi', 'macd', 'bb_', 'ema_', 'atr', 'vwap', 'obv', 'adx', 'stoch', 'cci', 'williams', 'mfi', 'tod_', 'dow_'))]
    fi_outputs = run_feature_importance_pipeline(
        frame=pred_df,
        feature_columns=feature_cols,
        output_dir='artifacts/analysis',
    )
    fi_outputs['combined'].head()
else:
    print('Prediction frame with feature columns not found. Skipping feature importance run.')

In [ ]:
# Step 3: run volatility-regime analysis and both backtest variants.
required_cols = {'timestamp', 'close', 'label', 'pred_label', 'pred_confidence', 'atr'}
if not pred_df.empty and required_cols.issubset(set(pred_df.columns)):
    eval_outputs = run_volatility_and_backtest_pipeline(
        predictions_frame=pred_df,
        output_dir='artifacts/analysis',
        confidence_threshold=0.6,
        risk_free_rate_annual=0.02,
    )
    eval_outputs['degradation']
else:
    print('Prediction frame missing required columns for volatility/backtest pipeline.')